In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.utils import resample
import timm

In [2]:
csv_path = "/kaggle/input/datasets/srimathisrahul/skin-image-dataset-dl/HAM10000_metadata.csv"
image_dir = "/kaggle/input/datasets/srimathisrahul/skin-image-dataset-dl/image_dataset-20260306T153013Z-1-001/image_dataset"

df_main = pd.read_csv(csv_path)

# remove normal
df_main = df_main[df_main['dx'] != 'normal'].reset_index(drop=True)

# create image path
df_main['image_path'] = df_main['image_id'].apply(
    lambda x: os.path.join(image_dir, x + ".jpg")
)

print("HAM dataset:", len(df_main))

HAM dataset: 10015


In [3]:
folder_path = "/kaggle/input/datasets/srimathisrahul/skinnewlabel/Train"

mapping = {
    "actinic keratosis": "akiec",
    "basal cell carcinoma": "bcc",
    "dermatofibroma": "df",
    "melanoma": "mel",
    "vascular lesion": "vasc"
}

data = []

for class_name in os.listdir(folder_path):
    class_folder = os.path.join(folder_path, class_name)

    if os.path.isdir(class_folder):
        for img in os.listdir(class_folder):
            data.append({
                "image_path": os.path.join(class_folder, img),
                "dx": mapping[class_name]
            })

df_new = pd.DataFrame(data)

print("Folder dataset:", len(df_new))

Folder dataset: 1229


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
label_map = {
    'nv':0,
    'mel':1,
    'bkl':2,
    'bcc':3,
    'akiec':4,
    'vasc':5,
    'df':6
}

df_main['label'] = df_main['dx'].map(label_map)
df_new['label'] = df_new['dx'].map(label_map)

In [5]:
df = pd.concat([df_main, df_new], ignore_index=True)

print("Total dataset:", len(df))
print(df['label'].value_counts())

Total dataset: 11244
label
0    6705
1    1567
2    1099
3     906
4     457
5     284
6     226
Name: count, dtype: int64


In [6]:
df['image_path'] = df['image_path'].apply(
    lambda x: x if os.path.exists(x) else None
)

print("Missing images:", df['image_path'].isnull().sum())

df = df[df['image_path'].notnull()].reset_index(drop=True)

print("After cleaning:", len(df))
print(df['label'].value_counts())

Missing images: 2303
After cleaning: 8941
label
0    5193
1    1266
2     850
3     794
4     394
5     244
6     200
Name: count, dtype: int64


In [7]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print("Train size:", len(train_df))
print("Val size:", len(val_df))

Train size: 7152
Val size: 1789


In [8]:
target_count = 2000

oversampled = []

for label in train_df['label'].unique():
    
    class_df = train_df[train_df['label'] == label]

    if len(class_df) < target_count:
        class_df = resample(
            class_df,
            replace=True,
            n_samples=target_count,
            random_state=42
        )

    oversampled.append(class_df)

train_df = pd.concat(oversampled)
train_df = train_df.sample(frac=1).reset_index(drop=True)

print("After Oversampling:")
print(train_df['label'].value_counts())
print("New train size:", len(train_df))

After Oversampling:
label
0    4154
2    2000
6    2000
5    2000
3    2000
4    2000
1    2000
Name: count, dtype: int64
New train size: 16154


In [9]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2,0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [10]:
class SkinDataset(Dataset):
    
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        
        img_path = self.df.iloc[idx]['image_path']
        label = self.df.iloc[idx]['label']

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [11]:
train_dataset = SkinDataset(train_df, train_transform)
val_dataset = SkinDataset(val_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [17]:
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import time

# -----------------------------
# MODEL
# -----------------------------
vit_model = timm.create_model(
    'vit_tiny_patch16_224',
    pretrained=True,
    num_classes=7,
    drop_rate=0.2
)

vit_model = vit_model.to(device)

# -----------------------------
# LOSS + OPTIMIZER
# -----------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    vit_model.parameters(),
    lr=3e-5,
    weight_decay=1e-4
)

# 🔥 Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.3, patience=2
)

# -----------------------------
# TRAINING SETTINGS
# -----------------------------
num_epochs = 15
best_val_loss = float('inf')

patience = 4
counter = 0

# -----------------------------
# TRAIN LOOP
# -----------------------------
for epoch in range(num_epochs):

    print(f"\n========== Epoch {epoch+1}/{num_epochs} ==========")
    start_time = time.time()

    # TRAINING
    vit_model.train()
    train_loss = 0
    total_batches = len(train_loader)

    for batch_idx, (images, labels) in enumerate(train_loader):

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = vit_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        if (batch_idx + 1) % 20 == 0 or (batch_idx + 1) == total_batches:
            print(f"[Train] Batch {batch_idx+1}/{total_batches} | Loss: {loss.item():.4f}")

    avg_train_loss = train_loss / total_batches

    # -----------------------------
    # VALIDATION
    # -----------------------------
    vit_model.eval()
    val_loss = 0
    correct = 0
    total = 0

    print("Running validation...")

    with torch.no_grad():
        for images, labels in val_loader:

            images, labels = images.to(device), labels.to(device)

            outputs = vit_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct / total

    # 🔥 scheduler step
    scheduler.step(avg_val_loss)

    epoch_time = time.time() - start_time

    print("\nEpoch Summary:")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}")
    print(f"Val Accuracy: {val_acc:.2f}%")
    print(f"Time: {epoch_time:.2f} sec")

    # -----------------------------
    # EARLY STOPPING
    # -----------------------------
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0

        torch.save(vit_model.state_dict(), "/kaggle/working/vit_bbb.pth")
        print("Model improved and saved!")

    else:
        counter += 1
        print(f" No improvement ({counter}/{patience})")

        if counter >= patience:
            print(" Early stopping triggered!")
            break


========== Epoch 1/15 ==========
[Train] Batch 20/505 | Loss: 1.8898
[Train] Batch 40/505 | Loss: 1.4389
[Train] Batch 60/505 | Loss: 1.5515
[Train] Batch 80/505 | Loss: 1.2980
[Train] Batch 100/505 | Loss: 1.3346
[Train] Batch 120/505 | Loss: 1.0644
[Train] Batch 140/505 | Loss: 1.1773
[Train] Batch 160/505 | Loss: 1.0623
[Train] Batch 180/505 | Loss: 1.0429
[Train] Batch 200/505 | Loss: 1.0273
[Train] Batch 220/505 | Loss: 0.9361
[Train] Batch 240/505 | Loss: 1.1483
[Train] Batch 260/505 | Loss: 1.0973
[Train] Batch 280/505 | Loss: 1.0196
[Train] Batch 300/505 | Loss: 1.0932
[Train] Batch 320/505 | Loss: 0.9887
[Train] Batch 340/505 | Loss: 0.9344
[Train] Batch 360/505 | Loss: 0.9609
[Train] Batch 380/505 | Loss: 0.7943
[Train] Batch 400/505 | Loss: 0.8454
[Train] Batch 420/505 | Loss: 0.9014
[Train] Batch 440/505 | Loss: 1.1340
[Train] Batch 460/505 | Loss: 0.8534
[Train] Batch 480/505 | Loss: 0.7611
[Train] Batch 500/505 | Loss: 0.6263
[Train] Batch 505/505 | Loss: 0.8319
Running 

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import time

# -----------------------------
# MODEL
# -----------------------------
mobilenet_model = models.mobilenet_v2(pretrained=True)

mobilenet_model.classifier[1] = nn.Linear(
    mobilenet_model.last_channel, 7
)

mobilenet_model = mobilenet_model.to(device)

print("\nMobileNetV2 model loaded!")

# -----------------------------
# LOSS + OPTIMIZER
# -----------------------------
criterion_m = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer_m = torch.optim.AdamW(
    mobilenet_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# 🔥 Scheduler
scheduler_m = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_m, mode='min', factor=0.3, patience=2
)

# -----------------------------
# TRAINING SETTINGS
# -----------------------------
num_epochs = 20
best_val_loss = float('inf')

patience = 4
counter = 0

# -----------------------------
# TRAIN LOOP
# -----------------------------
for epoch in range(num_epochs):

    print(f"\n========== MobileNet Epoch {epoch+1}/{num_epochs} ==========")
    start_time = time.time()

    # TRAINING
    mobilenet_model.train()
    train_loss = 0
    total_batches = len(train_loader)

    for batch_idx, (images, labels) in enumerate(train_loader):

        images, labels = images.to(device), labels.to(device)

        optimizer_m.zero_grad()

        outputs = mobilenet_model(images)
        loss = criterion_m(outputs, labels)

        loss.backward()
        optimizer_m.step()

        train_loss += loss.item()

        if (batch_idx + 1) % 20 == 0 or (batch_idx + 1) == total_batches:
            print(f"[Train] Batch {batch_idx+1}/{total_batches} | Loss: {loss.item():.4f}")

    avg_train_loss = train_loss / total_batches

    # -----------------------------
    # VALIDATION
    # -----------------------------
    mobilenet_model.eval()
    val_loss = 0
    correct = 0
    total = 0

    print("Running validation...")

    with torch.no_grad():
        for images, labels in val_loader:

            images, labels = images.to(device), labels.to(device)

            outputs = mobilenet_model(images)
            loss = criterion_m(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct / total

    # 🔥 scheduler step
    scheduler_m.step(avg_val_loss)

    epoch_time = time.time() - start_time

    print("\nMobileNet Epoch Summary:")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}")
    print(f"Val Accuracy: {val_acc:.2f}%")
    print(f"Time: {epoch_time:.2f} sec")

    # -----------------------------
    # EARLY STOPPING
    # -----------------------------
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0

        torch.save(mobilenet_model.state_dict(), "/kaggle/working/mobilenet_bbb.pth")
        print("MobileNet model improved and saved!")

    else:
        counter += 1
        print(f"No improvement ({counter}/{patience})")

        if counter >= patience:
            print("Early stopping triggered!")
            break

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 109MB/s] 



MobileNetV2 model loaded!

========== MobileNet Epoch 1/20 ==========
[Train] Batch 20/505 | Loss: 1.3284
[Train] Batch 40/505 | Loss: 1.3288
[Train] Batch 60/505 | Loss: 1.0914
[Train] Batch 80/505 | Loss: 1.0919
[Train] Batch 100/505 | Loss: 1.0007
[Train] Batch 120/505 | Loss: 0.9406
[Train] Batch 140/505 | Loss: 1.0910
[Train] Batch 160/505 | Loss: 1.0861
[Train] Batch 180/505 | Loss: 0.8618
[Train] Batch 200/505 | Loss: 0.8785
[Train] Batch 220/505 | Loss: 0.7619
[Train] Batch 240/505 | Loss: 0.9100
[Train] Batch 260/505 | Loss: 0.9046
[Train] Batch 280/505 | Loss: 1.0057
[Train] Batch 300/505 | Loss: 0.8801
[Train] Batch 320/505 | Loss: 0.8236
[Train] Batch 340/505 | Loss: 1.0739
[Train] Batch 360/505 | Loss: 0.6974
[Train] Batch 380/505 | Loss: 0.6854
[Train] Batch 400/505 | Loss: 0.9552
[Train] Batch 420/505 | Loss: 0.6690
[Train] Batch 440/505 | Loss: 0.8430
[Train] Batch 460/505 | Loss: 0.6294
[Train] Batch 480/505 | Loss: 0.8180
[Train] Batch 500/505 | Loss: 0.7188
[Train] 